<a href="https://colab.research.google.com/github/celina1astal/myf_first_project/blob/main/ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 📌 Install dependencies
!pip install -q langchain==1.0.5 langchain-community==0.4.1 langchain-groq==1.0.0 langchain-google-genai==3.0.1 chromadb==1.3.4 pypdf

In [ ]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "/content/scholarship_info.pdf"
loader = PyPDFLoader(file_path)
doc = loader.load()

In [ ]:
doc

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2025-08-07T19:16:29+05:30', 'author': 'Preethesh Poojary', 'moddate': '2025-08-07T19:16:29+05:30', 'source': '/content/scholarship_info.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Title: Scholarship Information 2025 \n \n1. Eligibility: \n- Open to students in India pursuing undergraduate degrees. \n- Annual family income must be below ₹6,00,000. \n- Minimum 60% marks in the last qualifying exam. \n \n2. Documents Required: \n- Income certificate \n- Aadhaar card \n- Bank passbook \n- Marksheet \n \n3. Deadline: October 15, 2025 \n \n4. Benefits: \n- ₹10,000 per semester for tuition \n- Book allowance of ₹3,000 per year \n \n5. How to Apply: \nVisit https://scholarships.gov.in and register under the NSP portal.')]

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=200
)

splits = splitter.split_documents(doc)
print(f"✅ Split into {len(splits)} chunks.")

# Preview first chunk
print("\n--- First Chunk ---\n")
print(splits[0].page_content[:500])

✅ Split into 2 chunks.

--- First Chunk ---

Title: Scholarship Information 2025 
 
1. Eligibility: 
- Open to students in India pursuing undergraduate degrees. 
- Annual family income must be below ₹6,00,000. 
- Minimum 60% marks in the last qualifying exam. 
 
2. Documents Required: 
- Income certificate 
- Aadhaar card 
- Bank passbook 
- Marksheet 
 
3. Deadline: October 15, 2025 
 
4. Benefits: 
- ₹10,000 per semester for tuition 
- Book allowance of ₹3,000 per year 
 
5. How to Apply:


storing data in vectordb

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_classic.vectorstores import Chroma

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GEMINI_API_KEY)

# Create Chroma vector store
vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="rag_chroma_db",
    collection_name="ipl_docs"
)

# Add documents
vector_store.add_documents(splits)

#vector_store
print(embeddings)

client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x7ccdd361a2d0> async_client=None model='models/gemini-embedding-001' task_type=None google_api_key=SecretStr('**********') credentials=None client_options=None base_url=None transport=None request_options=None


retriving

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

In [ ]:
from langchain_groq import ChatGroq
from google.colab import userdata
from langchain_groq import ChatGroq

# Load API key
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    temperature=0.3,
    max_tokens=200
)


In [ ]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)


In [ ]:
 #Query 1
query = "Who is the eligible for the scholarship?"
response = qa_chain.invoke({"query": query})

print("Query:", query)
print("Answer:", response["result"])

# Query 2
query2 = "What the benefits of the scholarship?"
response2 = qa_chain.invoke({"query": query2})

print("\nQuery:", query2)
print("Answer:", response2["result"])



Query: Who is the eligible for the scholarship?
Answer: **Eligible candidates**

- Students currently enrolled in an undergraduate program in India.  
- Their family’s annual income must be **below ₹6,00,000**.  
- They must have scored **at least 60 %** in their most recent qualifying examination (e.g., 12th grade, intermediate, or equivalent).

Query: What the benefits of the scholarship?
Answer: **Benefits of the scholarship**

- **₹10,000 per semester** to cover tuition fees.  
- **₹3,000 per year** as a book allowance.
